# Full networks — u/CaseA

Both networks over **every** alter, not just the 12 that share 2+ posts with the ego:

1. **Ego ↔ alter** — all 445 accounts that exchanged a direct reply with u/CaseA.
2. **Alter ↔ alter** — direct replies *between* those alters inside the ego's threads.

Inputs are the artifacts saved by `network_case_CaseA_ego_network_v1.ipynb`; nothing is re-pulled from
Arctic Shift. Account outcomes for the remaining alters are looked up with PRAW and cached into
the same `account_status.parquet`.

In [ ]:
# Libraries.
import pandas as pd
import numpy as np
import os
import re
import time
import glob
import networkx as nx
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from dotenv import load_dotenv
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)

# Case-study account, folders (reads the ego-network outputs), .env file for PRAW, and figure colours.
EGO = 'CaseA'
OUT_DIR = f'ego_net_{EGO}_output'
FIG_DIR = os.path.join('..', 'p2_figures')
ENV_PATH = '/home/eeporter/Reddit_project1_notebooks/reddit_env_vars.env'
CHECK_ACCOUNT_STATUS = True

SURFACE = '#fcfcfb'
MUTED = '#52514e'
INK = '#0b0b0b'
TIE_EDGE = '#7a7873'
EGO_EDGE = '#b5b3ad'
EGO_COLOR = '#0b0b0b'
# account-outcome colours taken from Figures/Reddit_figs/BlackPeopleTwitterclusters1.svg
STATUS_COLORS = {'Active': '#4b2362', 'Suspended': '#eeb182', 'Deleted': '#cfccc4',
                 'not checked': '#ffffff'}

# Every alter of the ego (no MIN_CONTACTS filter) and every ego-alter interaction.
alters_all = pd.read_parquet(os.path.join(OUT_DIR, f'alters_all_{EGO}.parquet'))
interactions = pd.read_parquet(os.path.join(OUT_DIR, f'interactions_{EGO}.parquet'))
print(f'alters: {len(alters_all):,}   interactions: {len(interactions):,}')
print(alters_all['contacts'].value_counts().sort_index().to_string())

## 1. Account outcomes for every alter
The ego notebook only looked up the 12 retained alters. The cache is extended to all 445.

In [ ]:
# Current account status (Active / Suspended / Deleted) from PRAW for the ego and ALL alters, cached to parquet
# (saved every 100 lookups). Credentials come from the .env file.
status_path = os.path.join(OUT_DIR, 'account_status.parquet')
account_status = dict(pd.read_parquet(status_path).values) if os.path.exists(status_path) else {}

if CHECK_ACCOUNT_STATUS:
    assert load_dotenv(ENV_PATH), f'could not load {ENV_PATH}'
    import praw, prawcore
    reddit_api = praw.Reddit(client_id=os.getenv('praw_script'),
                             client_secret=os.getenv('praw_key'),
                             username=os.getenv('login_name'),
                             password=os.getenv('password'),
                             user_agent=os.getenv('user_agent'),
                             ratelimit_seconds=360)
    to_check = [n for n in [EGO] + alters_all['alter'].tolist() if n not in account_status]
    print(f'{len(to_check):,} accounts to look up ({len(account_status):,} cached)')
    t_start = time.time()
    for k, name in enumerate(to_check, 1):
        retry_count = 0
        while retry_count < 10:
            try:
                redditor = reddit_api.redditor(name)
                try:
                    if hasattr(redditor, 'is_suspended'):
                        account_status[name] = 'Suspended'
                    elif hasattr(redditor, 'id'):
                        account_status[name] = 'Active'
                    else:
                        account_status[name] = 'Deleted'
                except prawcore.exceptions.NotFound:
                    account_status[name] = 'Deleted'
                break
            except prawcore.exceptions.TooManyRequests as e:
                retry_count += 1
                print(f'rate limited on {name}, retry {retry_count}')
                if retry_count == 10:
                    raise e
        if k % 100 == 0 or k == len(to_check):
            (pd.DataFrame(list(account_status.items()), columns=['account', 'account_status'])
               .to_parquet(status_path, engine='pyarrow'))
            print(f'  {k:,}/{len(to_check):,}  {(time.time() - t_start) / 60:.1f} min', flush=True)

# Attach status; counts overall and by number of shared posts with the ego.
alters_all['account_status'] = alters_all['alter'].map(account_status).fillna('not checked')
print(f'\nego status: {account_status.get(EGO, "not checked")}')
print(alters_all['account_status'].value_counts().to_string())
print('\nby shared posts with the ego:')
print(pd.crosstab(alters_all['contacts'], alters_all['account_status']).to_string())

## 2. Ego ↔ alter network — all alters
Ring layout: alters sit on rings by how many distinct posts they share with the ego (the innermost
ring is the strongest contact), ordered round each ring by account outcome so the suspended and
deleted accounts are not scattered at random. Node fill is the outcome, spoke width is shared posts.

In [ ]:
# Figure: every alter on rings by shared posts with the ego (strongest innermost), fill = account status.
def draw_ego_alter(alters, ego, figsize=(17, 17)):
    rings = sorted(alters['contacts'].unique(), reverse=True)     # strongest contact innermost
    radii = np.linspace(0.34, 1.0, len(rings)) if len(rings) > 1 else [0.7]
    order = {'Suspended': 0, 'Deleted': 1, 'Active': 2, 'not checked': 3}
    pos = {}
    for radius, ring in zip(radii, rings):
        members = (alters[alters['contacts'] == ring]
                   .assign(_o=lambda d: d['account_status'].map(order))
                   .sort_values(['_o', 'alter'])['alter'].tolist())
        for i, name in enumerate(members):
            ang = 2 * np.pi * i / len(members) + np.pi / 2
            r = radius if i % 2 == 0 else radius * 0.97    # stagger so neighbours do not touch
            pos[name] = np.array([r * np.cos(ang), r * np.sin(ang)])

    cmax = alters['contacts'].max()
    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
    for radius, ring in zip(radii, rings):
        ax.add_patch(plt.Circle((0, 0), radius, fill=False, ec='#e6e4df', lw=0.8, zorder=0))
        ax.annotate(f'{ring} shared post{"s" if ring > 1 else ""}', (-radius, 0), xytext=(-3, 0),
                    textcoords='offset points', ha='right', va='center', fontsize=8,
                    color=MUTED, zorder=5,
                    bbox=dict(boxstyle='round,pad=0.15', fc=SURFACE, ec='none', alpha=0.9))
    for _, r in alters.iterrows():
        x, y = pos[r['alter']]
        ax.plot([0, x], [0, y], color=EGO_EDGE, lw=0.25 + 0.75 * (r['contacts'] - 1) / max(cmax - 1, 1),
                alpha=0.3 + 0.45 * (r['contacts'] - 1) / max(cmax - 1, 1),
                solid_capstyle='round', zorder=1)
    ax.scatter([pos[a][0] for a in alters['alter']], [pos[a][1] for a in alters['alter']],
               s=[26 + 150 * (c - 1) / max(cmax - 1, 1) for c in alters['contacts']],
               c=[STATUS_COLORS[s] for s in alters['account_status']],
               edgecolors=MUTED, linewidths=0.55, zorder=3)
    ax.scatter([0], [0], s=700, c=EGO_COLOR, edgecolors=SURFACE, linewidths=2.5, zorder=4)
    ax.annotate(f'u/{ego}', (0, 0), xytext=(0, -16), textcoords='offset points', ha='center',
                va='top', fontsize=10, fontweight='bold', color=INK, zorder=6,
                bbox=dict(boxstyle='round,pad=0.2', fc=SURFACE, ec='none', alpha=0.95))
    # label only the repeat contacts: the 433 one-post alters cannot be labelled legibly, and the
    # ring is ordered by outcome so the suspended and deleted arcs are readable without names
    labelled = alters[alters['contacts'] > 1].sort_values(['contacts', 'alter'],
                                                          ascending=[False, True])
    for _, r in labelled.iterrows():
        x, y = pos[r['alter']]
        ang = np.arctan2(y, x)
        ax.annotate(f'{r["alter"]} ({r["contacts"]})', (x, y),
                    xytext=(11 * np.cos(ang), 11 * np.sin(ang)), textcoords='offset points',
                    ha='left' if np.cos(ang) >= 0 else 'right', va='center', fontsize=6.6,
                    color=INK, zorder=6,
                    bbox=dict(boxstyle='round,pad=0.1', fc=SURFACE, ec='none', alpha=0.85))
    mix = alters['account_status'].value_counts()
    handles = [Line2D([], [], marker='o', ls='', color=EGO_COLOR, markersize=11, label=f'ego (u/{ego})')]
    handles += [Line2D([], [], marker='o', ls='', mfc=STATUS_COLORS[st], mec=MUTED, mew=0.9,
                       markersize=9, label=f'{st}  {mix[st]} ({mix[st] / len(alters):.0%})')
                for st in ('Active', 'Suspended', 'Deleted', 'not checked') if mix.get(st)]
    handles.append(Line2D([], [], color=EGO_EDGE, lw=1.5, label='reply exchanged (width = shared posts)'))
    ax.legend(handles=handles, loc='upper left', frameon=False, fontsize=9)
    ax.set_title(f'u/{ego} ego–alter network — all {len(alters)} alters\n'
                 f'{len(labelled)} labelled: every alter sharing more than one post. '
                 f'Each ring is ordered by outcome, so the suspended and deleted accounts sit together.',
                 loc='left', fontsize=13, color=INK)
    ax.set_aspect('equal'); ax.axis('off'); ax.margins(0.06)
    fig.tight_layout()
    return fig


fig = draw_ego_alter(alters_all, EGO)
fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}_all_alters.png'), dpi=150,
            facecolor=SURFACE, bbox_inches='tight', pad_inches=0.3)
plt.show()

## 3. Alter ↔ alter ties, over all alters
A tie is a direct reply between two alters inside one of the ego's threads — the same definition
used for u/CaseB, but with no `MIN_CONTACTS` filter on the endpoints.

In [ ]:
# Alter-alter ties among ALL alters, from the cached thread pull: an alter replying to another alter's comment or post.
threads = (pd.concat([pd.read_parquet(f) for f in
                      sorted(glob.glob(os.path.join(OUT_DIR, 'thread_cache', 'threads_*.parquet')))],
                     ignore_index=True).drop_duplicates('id'))
thread_posts = pd.read_parquet(os.path.join(OUT_DIR, 'thread_post_authors.parquet'))
fullname_author = pd.concat([pd.Series(threads['author'].values, index='t1_' + threads['id']),
                             pd.Series(thread_posts['author'].values, index='t3_' + thread_posts['id'])])
fullname_author = fullname_author[~fullname_author.index.duplicated()]

# One row per unordered alter pair: distinct shared posts, replies, subreddits.
alter_set = set(alters_all['alter'])
th = threads.assign(parent_author=threads['parent_id'].map(fullname_author))
aa = th[th['author'].isin(alter_set) & th['parent_author'].isin(alter_set)
        & (th['author'] != th['parent_author'])]
alter_ties = (aa.assign(a=np.minimum(aa['author'], aa['parent_author']),
                        b=np.maximum(aa['author'], aa['parent_author']))
                .groupby(['a', 'b'])
                .agg(posts=('link_id', 'nunique'), replies=('id', 'size'),
                     subreddits=('subreddit', lambda s: ','.join(sorted(set(s)))))
                .reset_index())
assert (alter_ties['a'] < alter_ties['b']).all()
print(f'alter → alter replies: {len(aa):,}')
print(f'undirected ties: {len(alter_ties):,}   ties above one shared post: {(alter_ties["posts"] >= 2).sum()}')

# Undirected alter-alter graph; tied alters, connected components, isolated alters.
AT = nx.Graph()
AT.add_nodes_from(sorted(alter_set))
for _, r in alter_ties.iterrows():
    AT.add_edge(r['a'], r['b'], weight=int(r['posts']), n_replies=int(r['replies']),
                subreddits=r['subreddits'])
tied = sorted([n for n in AT.nodes() if AT.degree(n) > 0])
comps = sorted(nx.connected_components(AT.subgraph(tied)), key=lambda c: (-len(c), sorted(c)[0]))
print(f'alters with at least one tie: {len(tied)} of {len(alter_set)} ({len(tied)/len(alter_set):.1%})')
print(f'components: {len(comps)}   sizes: {[len(c) for c in comps]}')
print(f'isolated alters (no tie to another alter): {len(alter_set) - len(tied):,}')
alter_ties.sort_values(['posts', 'replies'], ascending=False).head(12)

In [ ]:
# Figure: alter-alter ties, each connected component laid out separately; fill = account status.
def draw_alter_ties(AT, tied, comps, alters, ego, figwidth=19):
    status = dict(zip(alters['alter'], alters['account_status']))
    contacts = dict(zip(alters['alter'], alters['contacts']))
    # components are tiny and disconnected: lay each one out separately and pack them into a grid
    n = len(comps)
    ncol = max(1, min(int(np.ceil(np.sqrt(n))), n, 6))
    nrow = int(np.ceil(n / ncol))
    figsize = (figwidth, max(4.0, 1.1 + nrow * figwidth / ncol * 0.42))
    pos, cell = {}, 1.0
    for k, c in enumerate(comps):
        sub = AT.subgraph(c)
        p = (nx.kamada_kawai_layout(sub) if len(c) > 2
             else {m: np.array([i - 0.5, 0.0]) for i, m in enumerate(sorted(c))})
        xy = np.array(list(p.values()))
        lo, span = xy.min(axis=0), max(np.ptp(xy, axis=0).max(), 1e-9)
        cx, cy = (k % ncol) * cell, -(k // ncol) * cell * 0.62
        scale = cell * (0.62 if len(c) > 3 else 0.42)
        for m, v in p.items():
            pos[m] = np.array([cx, cy]) + ((v - lo) / span - 0.5) * scale

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
    for u, v, d in AT.subgraph(tied).edges(data=True):
        ax.plot([pos[u][0], pos[v][0]], [pos[u][1], pos[v][1]], color=TIE_EDGE,
                lw=0.9 + 0.7 * (d['n_replies'] - 1), alpha=0.75, solid_capstyle='round', zorder=2)
    ax.scatter([pos[m][0] for m in tied], [pos[m][1] for m in tied],
               s=[70 + 60 * (contacts[m] - 1) for m in tied],
               c=[STATUS_COLORS[status[m]] for m in tied],
               edgecolors=MUTED, linewidths=0.8, zorder=3)
    for m in tied:
        x, y = pos[m]
        ax.annotate(m, (x, y), xytext=(0, 9), textcoords='offset points', ha='center', va='bottom',
                    fontsize=6.4, color=INK, zorder=6,
                    bbox=dict(boxstyle='round,pad=0.1', fc=SURFACE, ec='none', alpha=0.85))
    # one caption per component: where the tie was made
    for k, c in enumerate(comps):
        subs = sorted({s for _, _, d in AT.subgraph(c).edges(data=True)
                       for s in d['subreddits'].split(',')})
        cx, cy = (k % ncol) * cell, -(k // ncol) * cell * 0.62
        ax.annotate('r/' + ', r/'.join(subs[:2]), (cx, cy - cell * 0.24), ha='center', va='top',
                    fontsize=6.6, color=MUTED, zorder=6)
    mix = pd.Series([status[m] for m in tied]).value_counts()
    handles = [Line2D([], [], marker='o', ls='', mfc=STATUS_COLORS[st], mec=MUTED, mew=0.9,
                      markersize=9, label=f'{st}  {mix[st]} ({mix[st] / len(tied):.0%})')
               for st in ('Active', 'Suspended', 'Deleted', 'not checked') if mix.get(st)]
    handles.append(Line2D([], [], color=TIE_EDGE, lw=2, label='tie: a direct reply between two alters'))
    ax.legend(handles=handles, loc='lower right', frameon=False, fontsize=9)
    ax.set_title(f'u/{ego} alter–alter ties — all {len(alters)} alters considered\n'
                 f'{AT.subgraph(tied).number_of_edges()} ties among {len(tied)} alters in '
                 f'{len(comps)} unconnected groups; the other {len(alters) - len(tied)} alters have no tie. '
                 f'Every tie is a single shared post.',
                 loc='left', fontsize=13, color=INK, pad=20)
    ax.set_aspect('equal'); ax.axis('off'); ax.margins(0.12)
    fig.tight_layout()
    return fig


if comps:
    fig = draw_alter_ties(AT, tied, comps, alters_all, EGO)
    fig.savefig(os.path.join(FIG_DIR, f'ego_network_{EGO}_alter_ties_all.png'), dpi=150,
                facecolor=SURFACE, bbox_inches='tight', pad_inches=0.3)
    plt.show()
else:
    print(f'No alter-alter ties among any of the {len(alter_set)} alters: nothing to draw.')

## 4. Save

In [ ]:
# Save all alters (with status), all alter ties, the full ego-alter graph and the alter-tie graph; list the files.
alters_all.to_parquet(os.path.join(OUT_DIR, f'alters_all_{EGO}.parquet'),
                      engine='pyarrow', compression='zstd')
alter_ties.to_parquet(os.path.join(OUT_DIR, f'alter_ties_all_{EGO}.parquet'),
                      engine='pyarrow', compression='zstd')
EA = nx.Graph()
EA.add_node(EGO, account_status=account_status.get(EGO, 'not checked'), role='ego')
for _, r in alters_all.iterrows():
    EA.add_node(r['alter'], account_status=r['account_status'], role='alter',
                contacts=int(r['contacts']), replies=int(r['replies']))
    EA.add_edge(EGO, r['alter'], weight=int(r['contacts']), n_replies=int(r['replies']))
nx.write_graphml(EA, os.path.join(OUT_DIR, f'ego_alter_all_{EGO}.graphml'))
nx.set_node_attributes(AT, {n: account_status.get(n, 'not checked') for n in AT.nodes()}, 'account_status')
if tied:
    nx.write_graphml(AT.subgraph(tied), os.path.join(OUT_DIR, f'alter_ties_all_{EGO}.graphml'))
for f in sorted(glob.glob(os.path.join(OUT_DIR, '*all*'))):
    print(f'  {f}  ({os.path.getsize(f) / 1024:.1f} KB)')